# 02 - Particionamento oculto

## Objetivo

Demonstrar particionamento por mês usando `months(data_pagamento)` sem criar uma coluna técnica de partição.

## Valor para a PRODEMGE

Permite consultas mensais de pagamentos com melhor performance, mantendo o modelo de dados mais limpo para analistas.

## Como usar

1. Substitua os placeholders (`<datahub_link>`, `<usuário>` e `<senha>`) no primeiro bloco de código.
2. Execute a célula de configuração Livy.
3. Execute as células da demo.
4. No final, encerre a sessão Livy.

> Este notebook não executa Spark localmente. Todo código Spark SQL/PySpark é submetido ao Livy3 via REST API.

In [ ]:
import requests
import time
import json
import urllib3

# Remove warnings de certificado SSL self-signed
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =====================================================================
# CONFIGURAÇÕES DO LIVY3
# =====================================================================

LIVY_URL = "<datahub_link>"
USERNAME = "<usuário>"
PASSWORD = "<senha>"


BASE_LIVY_URL = LIVY_URL.rstrip("/")

# =====================================================================
# CONFIGURAÇÕES SPARK
# =====================================================================

SESSION_CONF = {
    "spark.app.name": "PRODEMGE_Iceberg_Demo_Livy3",

    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.executor.instances": "2",

    "spark.driver.memory": "2g",

    "spark.sql.catalog.iceberg_prod":
        "org.apache.iceberg.spark.SparkCatalog",

    "spark.sql.catalog.iceberg_prod.type":
        "hive",

    "spark.sql.extensions":
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
}

# =====================================================================
# CLIENTE HTTP
# =====================================================================

http = requests.Session()

http.auth = (USERNAME, PASSWORD)
http.verify = False

HEADERS = {
    "Content-Type": "application/json"
}

# =====================================================================
# AUXILIARES
# =====================================================================

def pretty_json(obj):
    print(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False
        )
    )

# =====================================================================
# CRIAÇÃO DA SESSÃO LIVY
# =====================================================================

def create_livy_session(
    kind="pyspark",
    conf=None,
    timeout_seconds=600
):

    payload = {
        "kind": kind,
        "conf": conf or SESSION_CONF
    }

    response = http.post(
        f"{BASE_LIVY_URL}/sessions",
        headers=HEADERS,
        data=json.dumps(payload),
        verify=False
    )

    response.raise_for_status()

    session_id = response.json()["id"]

    print(f"Sessão Livy criada: {session_id}")

    start = time.time()

    while True:

        response = http.get(
            f"{BASE_LIVY_URL}/sessions/{session_id}",
            verify=False
        )

        response.raise_for_status()

        payload = response.json()

        state = payload.get("state")

        print(f"Estado da sessão: {state}")

        if state == "idle":

            print(
                "Sessão Livy pronta para receber statements."
            )

            return session_id

        if state in ["dead", "error", "killed"]:

            pretty_json(payload)

            raise RuntimeError(
                f"Falha ao criar sessão Livy. Estado: {state}"
            )

        if time.time() - start > timeout_seconds:

            raise TimeoutError(
                "Timeout aguardando sessão Livy ficar idle."
            )

        time.sleep(5)

# =====================================================================
# EXECUÇÃO DE STATEMENTS
# =====================================================================

def submit_statement(
    session_id,
    code,
    kind="pyspark",
    timeout_seconds=900
):

    payload = {
        "code": code,
        "kind": kind
    }

    response = http.post(
        f"{BASE_LIVY_URL}/sessions/{session_id}/statements",
        headers=HEADERS,
        data=json.dumps(payload),
        verify=False
    )

    response.raise_for_status()

    statement_id = response.json()["id"]

    print(
        f"Statement submetido: {statement_id}"
    )

    start = time.time()

    while True:

        response = http.get(
            f"{BASE_LIVY_URL}/sessions/{session_id}/statements/{statement_id}",
            verify=False
        )

        response.raise_for_status()

        result = response.json()

        state = result.get("state")

        print(
            f"Estado do statement: {state}"
        )

        if state == "available":

            output = result.get(
                "output",
                {}
            )

            pretty_json(output)

            return output

        if state in [
            "error",
            "cancelling",
            "cancelled"
        ]:

            pretty_json(result)

            raise RuntimeError(
                f"Statement falhou. Estado: {state}"
            )

        if time.time() - start > timeout_seconds:

            raise TimeoutError(
                "Timeout aguardando statement finalizar."
            )

        time.sleep(3)

# =====================================================================
# ENCERRAMENTO DA SESSÃO
# =====================================================================

def close_livy_session(session_id):

    response = http.delete(
        f"{BASE_LIVY_URL}/sessions/{session_id}",
        verify=False
    )

    if response.status_code in [
        200,
        202,
        204
    ]:

        print(
            f"Sessão Livy encerrada: {session_id}"
        )

    else:

        print(
            f"Não foi possível encerrar a sessão {session_id}"
        )

        print(response.text)

# =====================================================================
# TESTE
# =====================================================================

livy_session_id = create_livy_session()

print(
    f"Sessão criada com sucesso: {livy_session_id}"
)

Sessão Livy criada: 7
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: idle
Sessão Livy pronta para receber statements.


In [3]:
# Normaliza o username para usar no nome da tabela sem caracteres inválidos.
table_suffix = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in USERNAME).strip("_")
table_name = f"beneficios_{table_suffix}"
particionado_table = f"beneficios_particionados_{table_suffix}"
staging_table = f"beneficios_staging_{table_suffix}"
csv_table = f"beneficios_csv_{table_suffix}"

spark_code = f"""
# =====================================================================
# PARTICIONAMENTO OCULTO COM ICEBERG
# =====================================================================

spark.sql(\"\"\"
CREATE DATABASE IF NOT EXISTS governo_mg
\"\"\")

spark.sql(\"\"\"
DROP TABLE IF EXISTS governo_mg.{particionado_table}
\"\"\")

# A tabela será fisicamente particionada por mês,
# mas o usuário consulta normalmente pela coluna data_pagamento.
spark.sql(\"\"\"
CREATE TABLE governo_mg.{particionado_table} (
    id_cidadao BIGINT,
    nome STRING,
    valor_beneficio DOUBLE,
    data_pagamento DATE,
    status STRING
)
USING iceberg
PARTITIONED BY (months(data_pagamento))
\"\"\")

spark.sql(\"\"\"
INSERT INTO governo_mg.{particionado_table} VALUES
(1, 'Carlos Pereira', 500.00, DATE '2025-01-01', 'ATIVO'),
(2, 'Fernanda Rocha', 700.00, DATE '2025-02-15', 'ATIVO'),
(3, 'Pedro Alves', 200.00, DATE '2025-03-10', 'SUSPENSO'),
(4, 'Juliana Costa', 800.00, DATE '2025-03-20', 'ATIVO')
\"\"\")

# Filtro por data. O Iceberg usa o particionamento oculto para pruning.
spark.sql(\"\"\"
SELECT *
FROM governo_mg.{particionado_table}
WHERE data_pagamento >= DATE '2025-02-01'
ORDER BY data_pagamento
\"\"\").show(truncate=False)

# Exibe metadados de arquivos para mostrar como os dados foram gravados.
spark.sql(\"\"\"
SELECT file_path, record_count
FROM governo_mg.{particionado_table}.files
\"\"\").show(truncate=False)

print("Particionamento oculto demonstrado com sucesso.")
"""

submit_statement(livy_session_id, spark_code)

Statement submetido: 0
Estado do statement: waiting
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: available
{
  "status": "ok",
  "execution_count": 0,
  "data": {
    "text/plain": "+----------+--------------+---------------+--------------+--------+\n|id_cidadao|nome          |valor_beneficio|data_pagamento|status  |\n+----------+--------------+---------------+--------------+--------+\n|2         |Fernanda Rocha|700.0          |2025-02-15    |ATIVO   |\n|3         |Pedro Alves   |200.0          |2025-03-10    |SUSPENSO|\n|4         |Juliana Costa |800.0          |2025-03-20    |ATIVO   |\n+----------+--------------+---------------+--------------+--------+\n\n+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+\n

{'status': 'ok',
 'execution_count': 0,
 'data': {'text/plain': '+----------+--------------+---------------+--------------+--------+\n|id_cidadao|nome          |valor_beneficio|data_pagamento|status  |\n+----------+--------------+---------------+--------------+--------+\n|2         |Fernanda Rocha|700.0          |2025-02-15    |ATIVO   |\n|3         |Pedro Alves   |200.0          |2025-03-10    |SUSPENSO|\n|4         |Juliana Costa |800.0          |2025-03-20    |ATIVO   |\n+----------+--------------+---------------+--------------+--------+\n\n+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+\n|file_path                                                                                                                                                                                          |record_count|\n+-------------------------

## Query equivalente no Hue / Impala

```sql
SELECT *
FROM governo_mg.beneficios_particionados
WHERE data_pagamento >= DATE '2025-02-01'
ORDER BY data_pagamento;
```

## Hidden Partitioning (Particionamento Oculto)

A tabela foi criada assim:
`PARTITIONED BY (months(data_pagamento))`

Então fisicamente o Iceberg organiza os arquivos por mês da data_pagamento.

Mas o usuário NÃO precisa consultar:
`WHERE mes = '2025-02'`

Nem precisa conhecer diretórios HDFS/S3.

Ele simplesmente faz:
`WHERE data_pagamento >= DATE '2025-02-01'`

E o Iceberg automaticamente:
- identifica os partitions relevantes
- faz partition pruning
- lê menos arquivos
- melhora performance

In [4]:
# Encerre a sessão ao final do notebook.
close_livy_session(livy_session_id)

Sessão Livy encerrada: 7
